# Prototype Pipeline for Validating LLM

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append('../..')

In [2]:
import boto3
import json
from typing import Dict, Any

class BedrockLLMClient:
    def __init__(self, region_name: str):
        """
        Initialize the AWS Bedrock client. Use `aws configure` first to add AWS_ACCESS_KEY_ID and AWS_SECRETE_ACCESS_KEY to the configuration 
        """
        self.client = boto3.client(
            'bedrock',
            region_name=region_name,
        )
    
    def send_prompt(self, model_id: str, prompt: str, parameters: Dict[str, Any] = None) -> str:
        """
        Send a prompt to the specified LLM model and retrieve the output message.
        
        :param model_id: The ID of the LLM model on AWS Bedrock.
        :param prompt: The input text prompt for the LLM.
        :param parameters: Additional parameters to control the LLM’s response.
        :return: The output text message from the LLM.
        """
        if parameters is None:
            parameters = {}
        
        try:
            # Construct the payload for the model inference request
            payload = {
                'modelId': model_id,
                'input': {
                    'prompt': prompt,
                    **parameters  # Optional parameters for tuning the response
                }
            }
            # Call Bedrock's `InvokeModel` API to send the prompt
            response = self.client.invoke_model(
                ContentType='application/json',
                Body=json.dumps(payload)
            )
            
            # Parse the JSON response from Bedrock
            response_body = json.loads(response['Body'].read().decode('utf-8'))
            return response_body.get('output', 'No output received')
        
        except Exception as e:
            print(f"An error occurred while sending the prompt: {e}")
            return str(e)


In [3]:
# # Example usage:
# if __name__ == "__main__":

# Set up AWS Bedrock client parameters
region = 'us-west-2'
model_id = 'anthropic.claude-3-haiku-20240307-v1:0'
prompt = "What is the capital of France?"

# Initialize the Bedrock LLM Client
bedrock_client = BedrockLLMClient(region_name=region)

# Send a prompt and get the output
output = bedrock_client.send_prompt(model_id=model_id, prompt=prompt)
print("Model Output:", output)

An error occurred while sending the prompt: 'Bedrock' object has no attribute 'invoke_model'
Model Output: 'Bedrock' object has no attribute 'invoke_model'


In [ ]:
from langchain_community.chat_models import BedrockChat

class AWSBedrock():
    def __init__(self, credentials_profile_name: str, region_name: str, endpoint_url: str, model_id: str, model_kwargs: dict):
        self.credentials_profile_name = credentials_profile_name
        self.region_name = region_name
        self.endpoint_url = endpoint_url
        self.model_id = model_id
        self.model_kwargs = model_kwargs
        self.model = self._initialize_model()

    def _initialize_model(self):
        return BedrockChat(
            credentials_profile_name=self.credentials_profile_name,
            region_name=self.region_name,
            endpoint_url=self.endpoint_url,
            model_id=self.model_id,
            model_kwargs=self.model_kwargs
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        response = chat_model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        response = await chat_model.ainvoke(prompt)
        return response.content

    def get_model_name(self):
        return self.model_id

# Example usage
# if __name__ == "__main__":
bedrock = AWSBedrock(
    credentials_profile_name="default",
    region_name="us-west-2",
    endpoint_url="https://bedrock-runtime.us-west-2.amazonaws.com",
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    model_kwargs={"temperature": 0.4}
)

prompt = "What is the weather like today?"
response = bedrock.generate(prompt)
print(response)

/var/folders/5d/_0rnq_b12f5bw3g4j6847zvc0000gp/T/ipykernel_8508/2505650274.py:13: LangChainDeprecationWarning: The class `BedrockChat` was deprecated in LangChain 0.0.34 and will be removed in 1.0. An updated version of the class exists in the langchain-aws package and should be used instead. To use it run `pip install -U langchain-aws` and import as `from langchain_aws import ChatBedrock`.
  return BedrockChat(


ValueError: Error raised by bedrock service: Error when retrieving token from sso: Token has expired and refresh failed

In [2]:
import boto3
import logging
from botocore.exceptions import ClientError
import json

class BedrockWrapper:
  
    def __init__(self,service,region):
        
        """ Initiates the bedrock client and runtime
        """
        self.bedrock_client = boto3.client(service_name=service, region_name=region)
        self.bedrock_runtime = boto3.client('bedrock-runtime')

    def list_foundation_models(self):
        """ List the foundational models available
        """
        response = self.bedrock_client.list_foundation_models()
        models = response["modelSummaries"]
        print(f"Got {len(models)} foundation models.", models)


    def set_model(self,model_id):
        """ sets the generative AI models Id to be used
        """
        self.model_id=model_id


    def generate_body(self,prompt,params):

        """ sets model parameter and prompt
        """

        body=json.dumps({
             'prompt': prompt,
             **params
        })

        return body

    def invoke_model(self,body):
        """calls the model and get response string
        """
      
        accept = 'application/json'
        contentType = 'application/json'
        response = self.bedrock_runtime.invoke_model(body=body, modelId= self.model_id, 
                                                     accept=accept, contentType=contentType)
        response_body = json.loads(response.get('body').read())
        result=response_body.get('completion')

        return result



if __name__ == "__main__":

    bedrock=BedrockWrapper("bedrock","us-weat-2")

    #bedrock.list_foundation_models()

    modelId = 'anthropic.claude-3-haiku-20240307-v1:0'

    bedrock.set_model(modelId)

    params={
        "max_tokens_to_sample": 1000,
        "temperature": 0.1,
        "top_p": 0.1,
    }  

    prompt=''' What is the weather today?'''
    
    body=bedrock.generate_body(prompt,params)

    result=bedrock.invoke_model(body)

    print(result)


TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed

In [2]:
import json
import os
import sys

import boto3

boto3_bedrock = boto3.client('bedrock')
[models['modelId'] for models in boto3_bedrock.list_foundation_models()['modelSummaries']]


TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed